<a href="https://colab.research.google.com/github/sangam34-same/AI-ML-Projects/blob/main/Project_1_SPR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image
import io, base64
from IPython.display import display, Javascript, JSON
from google.colab.output import eval_js
from google.colab import output
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

labels = ["rock", "paper", "scissors"]
data = {l: [] for l in labels}      # our empty photo album
print("Tools ready! GPU found:", len(tf.config.list_physical_devices('GPU')) > 0)

In [ ]:
CAPTURE_JS = """
async function captureClass(label, n, countdown, delay) {
  const sleep = ms => new Promise(r => setTimeout(r, ms));
  const BASE = 'https://cdn.jsdelivr.net/npm/@mediapipe/hands@0.4.1646424915/';
  const load = src => new Promise((res, rej) => {
    const s = document.createElement('script'); s.src = src; s.crossOrigin = 'anonymous';
    s.onload = res; s.onerror = rej; document.head.appendChild(s);
  });
  const box = document.createElement('div');
  const cv = document.createElement('canvas');
  cv.width = 640; cv.height = 480;
  const ctx = cv.getContext('2d');
  const video = document.createElement('video');
  video.style.display = 'none';
  box.append(video, cv);
  document.body.appendChild(box);
  google.colab.output.setIframeHeight(500, true);

  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
  video.srcObject = stream;
  await video.play();

  // hand checker (if it fails to load, we simply save every frame)
  let hands = null, hand = null, useTracker = false;
  try {
    if (!window.Hands) await load(BASE + 'hands.js');
    hands = new Hands({locateFile: f => BASE + f});
    hands.setOptions({maxNumHands: 1, modelComplexity: 0,
                      minDetectionConfidence: 0.6, minTrackingConfidence: 0.5});
    hands.onResults(r => {
      if (r.multiHandLandmarks && r.multiHandLandmarks.length) {
        const pts = r.multiHandLandmarks[0];
        const xs = pts.map(p => p.x * 640), ys = pts.map(p => p.y * 480);
        const x1 = Math.min(...xs), x2 = Math.max(...xs), y1 = Math.min(...ys), y2 = Math.max(...ys);
        hand = {cx: (x1 + x2) / 2, cy: (y1 + y2) / 2, r: Math.max(x2 - x1, y2 - y1) / 2 * 1.25};
      } else hand = null;
    });
    await hands.send({image: video});
    useTracker = true;
  } catch (e) { useTracker = false; }

  const small = document.createElement('canvas');
  small.width = 224; small.height = 224;
  const sctx = small.getContext('2d');

  let mode = 'countdown', num = countdown, count = 0, alive = true;

  function draw() {
    if (!alive) return;
    ctx.drawImage(video, 0, 0, 640, 480);
    ctx.textAlign = 'center';
    if (useTracker && hand) {
      ctx.beginPath(); ctx.arc(hand.cx, hand.cy, hand.r, 0, 6.2832);
      ctx.lineWidth = 5; ctx.strokeStyle = '#00e600'; ctx.stroke();
    }
    if (mode === 'countdown') {
      ctx.fillStyle = 'rgba(0,0,0,0.45)'; ctx.fillRect(0, 0, 640, 480);
      ctx.fillStyle = 'white'; ctx.font = 'bold 34px Arial';
      ctx.fillText('Get ready to show: ' + label.toUpperCase(), 320, 80);
      ctx.fillStyle = '#00e600'; ctx.font = 'bold 200px Arial';
      ctx.fillText(String(num), 320, 300);
      ctx.fillStyle = 'white'; ctx.font = '24px Arial';
      ctx.fillText('Recording starts at 0...', 320, 410);
    } else {
      ctx.textAlign = 'left';
      ctx.fillStyle = 'red'; ctx.font = 'bold 30px Arial';
      ctx.fillText('● RECORDING ' + label.toUpperCase() + '  ' + count + '/' + n, 15, 40);
      ctx.fillStyle = 'rgba(255,255,255,0.4)'; ctx.fillRect(15, 55, 610, 16);
      ctx.fillStyle = '#00e600'; ctx.fillRect(15, 55, 610 * count / n, 16);
      ctx.textAlign = 'center'; ctx.font = '22px Arial';
      if (useTracker && !hand) { ctx.fillStyle = '#ff5555'; ctx.fillText('Hand not found! Show your hand', 320, 465); }
      else { ctx.fillStyle = 'yellow'; ctx.fillText('Move slowly: closer, farther, tilt, left, right!', 320, 465); }
    }
    requestAnimationFrame(draw);
  }
  draw();

  const t0 = performance.now();
  while (performance.now() - t0 < countdown * 1000) {
    num = countdown - Math.floor((performance.now() - t0) / 1000);
    if (useTracker) { try { await hands.send({image: video}); } catch (e) {} }
    await sleep(60);
  }

  mode = 'recording';
  const frames = [];
  const rStart = performance.now();
  let last = 0;
  while (frames.length < n && performance.now() - rStart < 120000) {
    if (useTracker) { try { await hands.send({image: video}); } catch (e) {} }
    const now = performance.now();
    if ((!useTracker || hand) && now - last >= delay) {
      sctx.drawImage(video, 0, 0, 224, 224);
      frames.push(small.toDataURL('image/jpeg', 0.9));
      count = frames.length; last = now;
    }
    await sleep(20);
  }

  alive = false;
  stream.getTracks().forEach(t => t.stop());
  try { if (hands) hands.close(); } catch (e) {}
  box.remove();
  return frames;
}
"""

def capture(label, n=200, countdown=3, delay=150):
    display(Javascript(CAPTURE_JS))
    frames = eval_js(f'captureClass("{label}", {n}, {countdown}, {delay})')
    imgs = []
    for f in frames:
        img = Image.open(io.BytesIO(base64.b64decode(f.split(',')[1]))).convert('RGB')
        imgs.append(np.array(img))
    print(f"Captured {len(imgs)} photos of {label.upper()}")
    return imgs

print("Photo machine ready!")

In [ ]:
data["rock"] = capture("rock")

In [ ]:
data["paper"] = capture("paper")

In [ ]:
data["scissors"] = capture("scissors")

In [ ]:
for l in labels:
    c = len(data[l])
    print(f"{l:10s} {c} photos", "DONE" if c >= 100 else "⚠️ need at least 100, run its collect cell again")

fig, axes = plt.subplots(len(labels), 6, figsize=(14, 7))
for r, l in enumerate(labels):
    for c in range(6):
        idx = int(c * (len(data[l]) - 1) / 5)
        axes[r][c].imshow(data[l][idx])
        axes[r][c].set_title(l)
        axes[r][c].axis("off")
plt.tight_layout(); plt.show()

In [ ]:
X_tr, y_tr, X_va, y_va = [], [], [], []
for i, l in enumerate(labels):
    imgs = data[l]
    cut = int(len(imgs) * 0.8)          # first 80% = study, last 20% = exam
    X_tr += imgs[:cut];  y_tr += [i] * cut
    X_va += imgs[cut:];  y_va += [i] * (len(imgs) - cut)

X_train = np.array(X_tr, dtype="float32"); y_train = np.array(y_tr)
X_val   = np.array(X_va, dtype="float32"); y_val   = np.array(y_va)

print("Study photos:", len(X_train))
print("Exam photos:", len(X_val))

In [ ]:
augment = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.08),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
    tf.keras.layers.RandomBrightness(0.2, value_range=(0, 255)),
    tf.keras.layers.RandomContrast(0.2),
], name="augmentation")

base = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = augment(inputs)                                   # only active while training
x = tf.keras.layers.Rescaling(1/127.5, offset=-1)(x)  # numbers the brain likes
x = base(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(labels), activation="softmax")(x)

model = tf.keras.Model(inputs, outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
print("Brain built!")

In [ ]:
import time

# Safety check: fine-tuning needs the model from Cell 9 with the base brain
assert 'base' in globals() and 'model' in globals(), "Run Cell 9 (build the brain) first."

# Round 1 uses the early-stopping rule; Round 2 gets a fresh copy
def make_stop(patience):
    return tf.keras.callbacks.EarlyStopping(
        monitor="val_accuracy", patience=patience, restore_best_weights=True)

# Learning rate drops automatically when progress stalls
def make_lr():
    return tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=1, min_lr=1e-7, verbose=0)

# Fast data pipeline (keeps the GPU busy, so training is quicker)
AUTOTUNE = tf.data.AUTOTUNE
train_ds = (tf.data.Dataset.from_tensor_slices((X_train, y_train))
            .shuffle(len(X_train)).batch(32).prefetch(AUTOTUNE))
val_ds = (tf.data.Dataset.from_tensor_slices((X_val, y_val))
          .batch(32).prefetch(AUTOTUNE))

t0 = time.time()

# ---------- ROUND 1: teach only the new top layer ----------
print("ROUND 1: teaching the new top layer")
base.trainable = False
model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h1 = model.fit(train_ds, validation_data=val_ds, epochs=8,
               callbacks=[make_stop(3), make_lr()], verbose=2)

# ---------- ROUND 2: gently fine-tune the borrowed brain ----------
print("\nROUND 2: fine-tuning the last layers of the borrowed brain")
base.trainable = True
for layer in base.layers[:-30]:
    layer.trainable = False                       # only the last 30 layers adjust
for layer in base.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False                   # keeps training stable

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
h2 = model.fit(train_ds, validation_data=val_ds, epochs=6,
               callbacks=[make_stop(3), make_lr()], verbose=2)

print(f"\nTotal training time: {(time.time() - t0) / 60:.1f} minutes")
loss, acc = model.evaluate(val_ds, verbose=0)
print(f"Final exam accuracy: {acc * 100:.1f}%")

In [ ]:
def classify(data_url):
    try:
        img = Image.open(io.BytesIO(base64.b64decode(data_url.split(',')[1])))
        arr = np.array(img.convert('RGB').resize((224, 224)), dtype="float32")
        p = model(arr[np.newaxis, ...], training=False).numpy()[0]
        return JSON({"probs": [float(v) for v in p]})
    except Exception as e:
        return JSON({"error": str(e)})

output.register_callback('notebook.classify', classify)
print("AI is connected to the camera!")

In [ ]:
LIVE_JS = """
(async () => {
  const LABELS = %s;
  const MIN_CONF = 0.45, MIN_GAP = 0.12;
  const BASE = 'https://cdn.jsdelivr.net/npm/@mediapipe/hands@0.4.1646424915/';
  const load = src => new Promise((res, rej) => {
    const s = document.createElement('script'); s.src = src; s.crossOrigin = 'anonymous';
    s.onload = res; s.onerror = rej; document.head.appendChild(s);
  });
  await load(BASE + 'hands.js');

  const box = document.createElement('div');
  const btn = document.createElement('button');
  btn.textContent = '⏹ Stop camera';
  btn.style.cssText = 'font-size:16px;padding:8px 14px;margin-bottom:8px';
  const video = document.createElement('video'); video.style.display = 'none';
  const canvas = document.createElement('canvas'); canvas.width = 640; canvas.height = 480;
  const ctx = canvas.getContext('2d');
  box.append(btn, video, canvas); document.body.appendChild(box);
  google.colab.output.setIframeHeight(560, true);

  const stream = await navigator.mediaDevices.getUserMedia({video: {width: 640, height: 480}});
  video.srcObject = stream; await video.play();

  let hand = null, pts = null;
  const hands = new Hands({locateFile: f => BASE + f});
  hands.setOptions({maxNumHands: 1, modelComplexity: 1,
                    minDetectionConfidence: 0.6, minTrackingConfidence: 0.5});
  hands.onResults(r => {
    if (r.multiHandLandmarks && r.multiHandLandmarks.length) {
      pts = r.multiHandLandmarks[0];
      const xs = pts.map(p => p.x * 640), ys = pts.map(p => p.y * 480);
      const x1 = Math.min(...xs), x2 = Math.max(...xs), y1 = Math.min(...ys), y2 = Math.max(...ys);
      hand = {cx: (x1 + x2) / 2, cy: (y1 + y2) / 2, r: Math.max(x2 - x1, y2 - y1) / 2 * 1.25};
    } else { hand = null; pts = null; }
  });

  let history = [], avg = LABELS.map(() => 0), status = 'waiting for AI...', running = true;
  btn.onclick = () => { running = false; };
  const small = document.createElement('canvas'); small.width = 224; small.height = 224;
  const sctx = small.getContext('2d');
  const parse = res => {
    let d = res && res.data ? res.data['application/json'] : null;
    if (typeof d === 'string') { try { d = JSON.parse(d); } catch (e) {} }
    if (Array.isArray(d)) d = d[0]; return d;
  };

  // ask the AI only while a hand is visible
  (async () => {
    while (running) {
      if (hand) {
        try {
          sctx.drawImage(video, 0, 0, 224, 224);
          const res = await google.colab.kernel.invokeFunction(
            'notebook.classify', [small.toDataURL('image/jpeg', 0.9)], {});
          const d = parse(res);
          if (d && d.probs) {
            history.push(d.probs); if (history.length > 8) history.shift();
            avg = LABELS.map((_, i) => history.reduce((a, h) => a + h[i], 0) / history.length);
            status = 'AI is working';
          } else if (d && d.error) status = 'AI error: ' + d.error;
          else status = 'no reply from AI';
        } catch (e) { status = 'connection error: ' + e; }
      } else { history = []; avg = LABELS.map(() => 0); }
      await new Promise(r => setTimeout(r, 50));
    }
  })();

  const LINES = [[0,1],[1,2],[2,3],[3,4],[0,5],[5,6],[6,7],[7,8],[5,9],[9,10],[10,11],
                 [11,12],[9,13],[13,14],[14,15],[15,16],[13,17],[17,18],[18,19],[19,20],[0,17]];

  while (running) {
    try { await hands.send({image: video}); } catch (e) { status = 'hand tracker error: ' + e; }
    try {
      ctx.drawImage(video, 0, 0, 640, 480);
      const order = avg.map((v, i) => i).sort((a, b) => avg[b] - avg[a]);
      const best = order[0], conf = avg[best], gap = conf - avg[order[1]];
      ctx.textAlign = 'left';

      if (hand) {
        ctx.strokeStyle = 'rgba(0,230,0,0.6)'; ctx.lineWidth = 2;
        LINES.forEach(([a, b]) => { ctx.beginPath();
          ctx.moveTo(pts[a].x * 640, pts[a].y * 480); ctx.lineTo(pts[b].x * 640, pts[b].y * 480); ctx.stroke(); });
        ctx.beginPath(); ctx.arc(hand.cx, hand.cy, hand.r, 0, 6.2832);
        ctx.lineWidth = 6; ctx.strokeStyle = '#00e600'; ctx.stroke();

        const sure = conf >= MIN_CONF && gap >= MIN_GAP;
        const name = history.length === 0 ? '...' : (sure ? LABELS[best].toUpperCase() : 'NOT SURE');
        const text = name + ' ' + Math.round(conf * 100) + '%%';
        ctx.font = 'bold 34px Arial';
        const w = ctx.measureText(text).width;
        const tx = Math.max(10, Math.min(hand.cx - w / 2, 630 - w)), ty = Math.max(45, hand.cy - hand.r - 15);
        ctx.fillStyle = sure ? '#00e600' : '#ffcc00'; ctx.fillRect(tx - 8, ty - 36, w + 16, 46);
        ctx.fillStyle = 'black'; ctx.fillText(text, tx, ty);
      } else {
        // no hand on screen: automatically show NOTHING
        ctx.font = 'bold 34px Arial';
        ctx.fillStyle = 'rgba(80,80,80,0.85)'; ctx.fillRect(15, 10, 190, 46);
        ctx.fillStyle = 'white'; ctx.fillText('NOTHING', 25, 46);
      }

      ctx.font = '16px Arial';
      LABELS.forEach((l, i) => {
        const y = 60 + i * 30;
        ctx.fillStyle = 'rgba(0,0,0,0.55)'; ctx.fillRect(470, y - 16, 160, 24);
        ctx.fillStyle = (hand && i === best) ? '#00e600' : '#aaaaaa';
        ctx.fillRect(472, y - 14, 156 * avg[i], 20);
        ctx.fillStyle = 'white'; ctx.fillText(l + ' ' + Math.round(avg[i] * 100) + '%%', 476, y + 2);
      });
      ctx.fillStyle = 'yellow'; ctx.fillText(status, 10, 470);
    } catch (e) { status = 'draw error: ' + e; }
    await new Promise(r => requestAnimationFrame(r));
  }
  stream.getTracks().forEach(t => t.stop()); box.remove();
})();
""" % str(labels)

display(Javascript(LIVE_JS))

In [ ]:
def add_more(label, n=200):
    data[label] += capture(label, n=n)
    print(label, "now has", len(data[label]), "photos")

# Example: the AI mixes up scissors. Add scissors photos from a new distance, a new
# background, or a friend's hand. Uncomment the line below to run it:
# add_more("scissors")